# Analisis rapido abandono - v3 (final) (2)

Notas para mi: notebook para ver si el modelo funciona antes de la demo del jueves.
No tocar mucho el orden de las celdas, ya casi esta. Lo dejo tal cual para ensenar
en la reunion.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [ ]:
# Intento cargar un archivo con datos adicionales de otra campana.
# Esta ruta es la de mi portatil, la dejo asi de momento.
RUTA_EXTRA = "/Users/martacosta/proyectos/churn_demo/datos/clientes_extra.csv"
try:
    extra = pd.read_csv(RUTA_EXTRA)
    print("datos extra cargados:", extra.shape)
except FileNotFoundError:
    extra = None
    print("no se encontro el archivo extra, seguimos sin el")

In [ ]:
# Genero un dataset sintetico parecido al de churn para probar el modelo.
n = 400
datos = pd.DataFrame({
    "antiguedad_meses": np.random.randint(1, 72, n),
    "uso_mensual_horas": np.random.gamma(2, 5, n),
    "tickets_soporte": np.random.poisson(1.5, n),
    "factura_mensual": np.random.normal(35, 12, n),
})
prob_base = 0.05 + 0.5 * (datos["tickets_soporte"] > 2) + 0.5 * (datos["uso_mensual_horas"] > 18)
datos["abandono"] = (np.random.rand(n) < prob_base.clip(0.02, 0.95)).astype(int)

# Si el archivo "extra" existe, le sumamos una columna suya como feature mas.
# A nadie mas le va a existir ese archivo, asi que en la practica esta columna
# es casi siempre 0 - pero eso no se ve a simple vista, solo ejecutando.
if extra is not None and "nivel_regional" in extra.columns:
    datos["nivel_regional"] = extra["nivel_regional"].values[:n]
else:
    datos["nivel_regional"] = 0

print(datos.shape)
datos.head()

### Vista rapida

No hace falta comprobar nulos ni rangos raros: el dataset ya viene limpio de la fuente.

In [ ]:
# Umbral de probabilidad a partir del cual consideramos que un cliente esta "en riesgo".
umbral_riesgo = 0.5

Probando algo rapido: si bajo (o subo) el umbral, a ver cuantos clientes en riesgo salen.
Luego lo dejo como estaba... o no, ya se vera.

In [ ]:
# Prueba rapida con un umbral mas exigente.
umbral_riesgo = 0.8

In [ ]:
datos["prob_estimada"] = prob_base
clientes_riesgo = datos[datos["prob_estimada"] > umbral_riesgo]
print("umbral usado:", umbral_riesgo, "- clientes en riesgo:", len(clientes_riesgo))

In [ ]:
X = datos[["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual", "nivel_regional"]]
y = datos["abandono"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [ ]:
modelo = LogisticRegression(max_iter=500)
modelo.fit(X_train_scaled, y_train)  # ojo: X_train_scaled se define mas abajo, en la celda del escalado
print("modelo entrenado")

### Escalado

Se me olvido escalar antes de entrenar. Lo dejo aqui de momento, ya lo reordenare.

In [ ]:
escalador = StandardScaler()
X_train_scaled = escalador.fit_transform(X_train)
X_test_scaled = escalador.transform(X_test)

In [ ]:
precision_train = modelo.score(X_train_scaled, y_train)
precision_test = modelo.score(X_test_scaled, y_test)
print("precision train:", precision_train, "- precision test:", precision_test)

### Guardar el modelo

In [ ]:
RUTA_MODELO = "/Users/martacosta/proyectos/churn_demo/modelos/modelo_final.pkl"
# joblib.dump(modelo, RUTA_MODELO)  # ya lo guardare cuando tenga la version buena
print("Fin del notebook. El modelo queda solo en memoria: se pierde al cerrar Jupyter.")